# Dictionaries with a key that is not a string

BHoM writes a C# dictionary whose key is not a string (an enum, a `Guid`, a class) as an **array of `{"k": key, "v": value}` objects**. This notebook checks that `bhom_schema` reads that form, writes it back, refuses a wrong enum key, and describes it in its JSON Schema.

Run the cells from the top, in order. Each cell says what to expect; anything else is a bug to report. The classes are defined here, so only the package is needed, and it must be a build that has the fix (cell 1 says whether it does).

## 1. Is the package the right one?

Run the cell. It prints the version, and the import fails with `ModuleNotFoundError` on a package that is older than the fix. In that case upgrade (see the guide) and restart the kernel.

In [1]:
import bhom_schema
print(bhom_schema.__version__)

from bhom_schema._support.key_value_pairs import keyed_fields
print("the package has the fix")

10.0.0.dev0
the package has the fix


## 2. Define a class that has such dictionaries

`Support` is an enum with a member that C# calls `None` (Python writes it `None_`). `Totals` has a dictionary keyed by the enum, and a dictionary keyed by a class (`Point`), which Python holds as a list of pairs, with the C# type recorded in a `ClrType` marker.

Expect no output.

In [2]:
import json
from typing import Annotated, Any, ClassVar

from pydantic import ValidationError

from bhom_schema._support import BHoMEnum, ClrType, SchemaModel

class Support(BHoMEnum):
    __bhom__: ClassVar[dict[str, Any]] = {"clr_name": "Demo.Support", "kind": "enum", "renamed_members": {"None_": "None"}}
    None_ = 0
    Fixed = 1
    Pinned = 2

class Point(SchemaModel):
    __bhom__: ClassVar[dict[str, Any]] = {"clr_name": "Demo.Point", "kind": "class"}
    X: float = 0.0

class Totals(SchemaModel):
    __bhom__: ClassVar[dict[str, Any]] = {"clr_name": "Demo.Totals", "kind": "class"}
    ByKind: dict[Support, float] = {}
    ByPoint: Annotated[list[tuple[Point, str]], ClrType("System.Collections.Generic.Dictionary<Demo.Point, string>")] = []

## 3. Read the JSON that C# writes

The text below has the form that the BHoM serialiser writes for these two dictionaries (the first is the real output of `Serialiser_Engine`, the second follows the same rule).

Expect `{<Support.Fixed: 1>: 2.5, <Support.None_: 0>: 1.0}` and `[(Point(bhom_version=None, X=1.0), 'one')]`. Before the fix this cell failed with `Input should be a valid dictionary`.

In [3]:
CSHARP_JSON = (
    '{ "_t" : "Demo.Totals", "ByKind" : [{ "k" : "Fixed", "v" : 2.5 }, { "k" : "None", "v" : 1.0 }], '
    '"ByPoint" : [{ "k" : { "X" : 1.0 }, "v" : "one" }] }')

totals = Totals.model_validate_json(CSHARP_JSON)
print(totals.ByKind)
print(totals.ByPoint)

{<Support.Fixed: 1>: 2.5, <Support.None_: 0>: 1.0}
[(Point(bhom_version=None, X=1.0), 'one')]


## 4. Write it back

Expect both dictionaries as arrays of `k`/`v` objects, with the C# member name `None` (not `None_`) as a key, the `Point` key of `ByPoint` written as an object with its `_t`, and then `True`: what was written reads back to an equal object. Before the fix `ByKind` was written as an object, `{"Fixed": 2.5, "None": 1.0}`.

In [4]:
written = totals.model_dump_json(indent=2)
print(written)
print(Totals.model_validate_json(written) == totals)

{
  "_t": "Demo.Totals",
  "ByKind": [
    {
      "k": "Fixed",
      "v": 2.5
    },
    {
      "k": "None",
      "v": 1.0
    }
  ],
  "ByPoint": [
    {
      "k": {
        "_t": "Demo.Point",
        "X": 1.0
      },
      "v": "one"
    }
  ]
}
True


## 5. The older form is still read

A dictionary written as a plain JSON object (older Python output, or a hand-written file) is still accepted. Expect `{<Support.Fixed: 1>: 1.0, <Support.None_: 0>: 2.0}`.

In [5]:
print(Totals.model_validate({"ByKind": {"Fixed": 1.0, "None": 2.0}}).ByKind)

{<Support.Fixed: 1>: 1.0, <Support.None_: 0>: 2.0}


## 6. A key that is not a member is flagged

The BHoM serialiser does not flag this: in C# the key silently becomes the member with value 0. Expect two `ValidationError` messages that say `Fixd is not a member of Support. Members: None, Fixed, Pinned. Did you mean Fixed?`, one for each way of giving the key.

In [6]:
for bad in ({"ByKind": {"Fixd": 1.0}}, {"ByKind": [{"k": "Fixd", "v": 1.0}]}):
    try:
        Totals.model_validate(bad)
        print("NOT FLAGGED")
    except ValidationError as error:
        print(error.errors()[0]["msg"])

Fixd is not a member of Support. Members: None, Fixed, Pinned. Did you mean Fixed?
Fixd is not a member of Support. Members: None, Fixed, Pinned. Did you mean Fixed?


## 7. The JSON Schema

The schema describes the array, as BHoM_JSONSchema does, and the JSON of step 3 validates against it. This needs the `jsonschema` package; if the import fails, run `%pip install jsonschema` in a cell and run this one again.

Expect `array`, then `[]` (no errors for the C# JSON), then a non-empty list (the old object form is not what the schema describes).

In [7]:
from jsonschema import Draft202012Validator

schema = Totals.model_json_schema()
print(schema["properties"]["ByKind"]["type"])
print(list(Draft202012Validator(schema).iter_errors(json.loads(CSHARP_JSON))))
print(len(list(Draft202012Validator(schema).iter_errors({"ByKind": {"Fixed": 2.5}}))) > 0)

array
[]
True


## 8. Summary

Expect `All checks passed`.

In [8]:
assert totals.ByKind == {Support.Fixed: 2.5, Support.None_: 1.0}
assert [(point.X, name) for point, name in totals.ByPoint] == [(1.0, "one")]
assert json.loads(written)["ByKind"] == [{"k": "Fixed", "v": 2.5}, {"k": "None", "v": 1.0}]
assert json.loads(written)["ByPoint"][0]["k"]["X"] == 1.0
assert Totals.model_validate_json(written) == totals
print("All checks passed")

All checks passed
